# Qwen3.5 Audit v11 CVS Context v3.2

This notebook derives `v3.2` from cached `cvsctx_v3__debug5` results, then runs a separate interval-level right-tool rubric over the predicted right-hand action intervals.

Workflow:
1. Load cached `v3` results.
2. For every predicted right-hand interval in every method row, run and cache the best rubric prompt separately.
3. Recombine into `v3.2` with a narrow rule: if the original right interval tool is `Hook` and the interval rubric predicts `Maryland`, rewrite that interval tool to `Maryland`.

The interval rubric outputs are saved separately so they can be recombined with different policies later.


In [1]:
from openai import OpenAI
import base64
import hashlib
import json
import os
import re
import sys
from copy import deepcopy
from pathlib import Path

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cvs_act.action_segment_eval import (
    convert_record_to_simple_actions,
    evaluate_method_predictions,
    extract_json_object,
    load_audit_records,
    naturalize_simple_actions,
    read_json,
    write_json,
)

with open('/mnt/md0/weiqiuy/ips/carnaroli.txt') as input_file:
    carnaroli_ip = input_file.read().strip()
client = OpenAI(base_url=f'http://{carnaroli_ip}:8001/v1', api_key=os.environ['OPENAI_API_KEY'])

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B-FP8'
REQUEST_TEMPERATURE = 0
REQUEST_ENABLE_THINKING = False
PROMPT_VERSION = '2026-05-23-v3_2-right-interval-rubric-v1'
FRAME_STRIDE = 30
ANNOTATION_ROOT = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1'
AUDIT_V11_DIR = ANNOTATION_ROOT / 'audit_v11'
FRAMES_DIR = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/frames/test'
SOURCE_ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5'
ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_2'
INTERVAL_CACHE_PATH = ARTIFACT_DIR / 'right_interval_rubric_cache.json'
INTERVAL_ROWS_PATH = ARTIFACT_DIR / 'right_interval_rubric_rows.json'
RESULTS_PATH = ARTIFACT_DIR / 'results.json'
SPEC_EVAL_PATH = ARTIFACT_DIR / 'spec_eval_results.json'
SUMMARY_PATH = ARTIFACT_DIR / 'override_summary.json'
LABELING_VERSION = 'code_labels_v3_2_interval_rubric_hook_to_maryland'
WRITE_DERIVED_ARTIFACTS = True
FORCE_INTERVAL_RUBRIC = False

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('source:', SOURCE_ARTIFACT_DIR)
print('artifacts:', ARTIFACT_DIR)


source: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5
artifacts: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_2__debug5


In [2]:
REFERENCE_IMAGE_PATHS = {
    'holes': ROOT_DIR / 'few_shot_examples/shape/2d/holes_vs_no_holes.png',
    'sharp': ROOT_DIR / 'few_shot_examples/shape/2d/sharp_vs_blunt.png',
    'tapered': ROOT_DIR / 'few_shot_examples/shape/2d/tapered_vs_constant.png',
    'tube': ROOT_DIR / 'few_shot_examples/shape/2d/tube_vs_with_tiip.png',
    'two_tips': ROOT_DIR / 'few_shot_examples/shape/2d/two_tips_vs_one.png',
    'wire': ROOT_DIR / 'few_shot_examples/shape/2d/wire_vs_not_wire.png',
    'hollow': ROOT_DIR / 'few_shot_examples/shape/2d/hollow_vs_solid_tip.png',
    'hinge': ROOT_DIR / 'few_shot_examples/shape/2d/hinge_vs_no_hinge.png',
    'serration': ROOT_DIR / 'few_shot_examples/shape/2d/serration_vs_smooth.png',
    'pivot': ROOT_DIR / 'few_shot_examples/shape/2d/pivot_vs_parallel.png',
}

instrument_rubric = [
    {'id': 1, 'question': 'Does the distal tube have hole-like openings?', 'description': {'yes': 'Yes if there are one or more visible holes on the tube/shaft.', 'no': 'No if the tube/shaft surface does not have holes.', 'uncertain': 'Uncertain if the end of the tube/shaft is obscured or there is no visible tool.'}, 'reference_name': 'holes', 'reference_description': 'Visible hole-like openings versus a smooth tube.', 'scores': {'Grasper': ('No', 2), 'Maryland': ('No', 2), 'Hook': ('No', 2), 'Irrigator': ('Yes', 2), 'Scissors': ('No', 2), 'Clipper': ('No', 2)}},
    {'id': 2, 'question': 'Does the tip (each tip, if multiple) have a cutting edge (blade)?', 'description': {'yes': 'Yes if visible sharp cutting edges or blades are present.', 'no': 'No if the tips are blunt.', 'uncertain': 'Uncertain if the tips are obscured.'}, 'reference_name': 'sharp', 'reference_description': 'Sharp cutting edge versus blunt tip.', 'scores': {'Grasper': ('No', 2), 'Maryland': ('No', 2), 'Hook': (None, 0), 'Irrigator': ('No', 2), 'Scissors': ('Yes', 2), 'Clipper': ('No', 2)}},
    {'id': 3, 'question': 'Is the metal tip of the tool (each tip, if multiple) tapered?', 'description': {'yes': 'Yes if the width decreases smoothly toward the tip.', 'no': 'No if the width stays mostly constant.', 'uncertain': 'Uncertain if the tip is obscured.'}, 'reference_name': 'tapered', 'reference_description': 'Tapered versus non-tapered tip.', 'scores': {'Grasper': ('No', 2), 'Maryland': ('Yes', 2), 'Hook': ('No', 2), 'Irrigator': ('No', 2), 'Scissors': ('Yes', 2), 'Clipper': (None, 0)}},
    {'id': 4, 'question': 'Does the distal end consist of a tube only (without any separate tip structure)?', 'description': {'yes': 'Yes if the end is just a tube.', 'no': 'No if there is a distinct tip structure.', 'uncertain': 'Uncertain if the distal end is obscured.'}, 'reference_name': 'tube', 'reference_description': 'Tube-only distal end versus a distinct tip structure.', 'scores': {'Grasper': ('No', 2), 'Maryland': ('No', 2), 'Hook': ('No', 2), 'Irrigator': ('Yes', 2), 'Scissors': ('No', 2), 'Clipper': ('No', 2)}},
    {'id': 5, 'question': 'Does the tip split into two visible parts?', 'description': {'yes': 'Yes if two separate tips or jaws are visible.', 'no': 'No if only a single continuous tip is visible.', 'uncertain': 'Uncertain if the tip is obscured.'}, 'reference_name': 'two_tips', 'reference_description': 'Two visible tips or jaws versus a single continuous tip.', 'scores': {'Grasper': ('Yes', 1), 'Maryland': ('Yes', 1), 'Hook': ('No', 1), 'Irrigator': ('No', 1), 'Scissors': ('Yes', 1), 'Clipper': ('Yes', 1)}},
    {'id': 6, 'question': 'Does the tip appear wire-like (very thin and rod-like, much thinner than typical jaws or blades)?', 'description': {'yes': 'Yes if the tip resembles a thin metal wire.', 'no': 'No if the tip is broader or thicker.', 'uncertain': 'Uncertain if the tip is obscured.'}, 'reference_name': 'wire', 'reference_description': 'Wire-like tip versus thicker tip.', 'scores': {'Grasper': ('No', 2), 'Maryland': ('No', 2), 'Hook': ('Yes', 2), 'Irrigator': ('No', 2), 'Scissors': ('No', 2), 'Clipper': ('No', 2)}},
    {'id': 7, 'question': 'Does the tip appear hollow (having an opening or lumen) rather than solid?', 'description': {'yes': 'Yes if an opening or inner cavity is visible.', 'no': 'No if the tip appears solid.', 'uncertain': 'Uncertain if the tip is obscured.'}, 'reference_name': 'hollow', 'reference_description': 'Hollow tip versus solid tip.', 'scores': {'Grasper': ('Yes', 2), 'Maryland': ('No', 2), 'Hook': ('No', 2), 'Irrigator': ('No', 2), 'Scissors': ('No', 2), 'Clipper': ('No', 2)}},
    {'id': 8, 'question': 'Is a hinge (pivot joint) visible where the tip is attached to the shaft?', 'description': {'yes': 'Yes if a visible pivot or joint is present.', 'no': 'No if no pivot is visible.', 'uncertain': 'Uncertain if the hinge region is obscured.'}, 'reference_name': 'hinge', 'reference_description': 'Visible hinge or pivot versus none.', 'scores': {'Grasper': ('Yes', 1), 'Maryland': ('Yes', 1), 'Hook': ('No', 1), 'Irrigator': ('No', 1), 'Scissors': ('Yes', 1), 'Clipper': ('Yes', 1)}},
    {'id': 9, 'question': 'Do any visible tips have serrations (small, repeated tooth-like ridges along the edge)?', 'description': {'yes': 'Yes if a serrated edge is visible.', 'no': 'No if the edge looks smooth.', 'uncertain': 'Uncertain if the edge is obscured.'}, 'reference_name': 'serration', 'reference_description': 'Serrated edge versus smooth edge.', 'scores': {'Grasper': ('Yes', 1), 'Maryland': ('Yes', 2), 'Hook': ('No', 2), 'Irrigator': ('No', 2), 'Scissors': ('No', 2), 'Clipper': ('No', 2)}},
    {'id': 10, 'question': 'Do the two tips originate from a common pivot point at the base (opening like a fan)?', 'description': {'yes': 'Yes if both tips meet at a single joint and spread apart from that point.', 'no': 'No if the tips remain parallel.', 'uncertain': 'Uncertain if the base is obscured.'}, 'reference_name': 'pivot', 'reference_description': 'Fan-opening common pivot versus parallel tips.', 'scores': {'Grasper': ('Yes', 2), 'Maryland': ('Yes', 2), 'Hook': (None, 0), 'Irrigator': (None, 0), 'Scissors': ('Yes', 2), 'Clipper': ('No', 2)}},
]

frame_checks = [
    {'id': 1, 'question': 'Is the active working instrument present in the frame?', 'description': {'yes': 'Yes if the active working instrument is visible.', 'no': 'No if it is clearly absent.', 'uncertain': 'Uncertain if visibility is too poor.'}},
    {'id': 2, 'question': 'Is the tip of the actively working instrument obscured by tissue?', 'description': {'yes': 'Yes if tissue covers the tip.', 'no': 'No if the tip is visible.', 'uncertain': 'Uncertain if the tip cannot be localized.'}},
    {'id': 3, 'question': 'Is the tip of the actively working instrument blurry?', 'description': {'yes': 'Yes if the tip is blurry.', 'no': 'No if the tip is sharp enough to inspect.', 'uncertain': 'Uncertain if the tip cannot be localized.'}},
]

TOOL_TYPES = ['Grasper', 'Maryland', 'Hook', 'Irrigator', 'Scissors', 'Clipper']
QUESTION_SPECS = []
for item in instrument_rubric:
    QUESTION_SPECS.append({'qid': f"instrument_{item['id']}", 'group': 'instrument', 'question': item['question'], 'description': item['description'], 'reference_name': item['reference_name'], 'reference_description': item['reference_description']})
for item in frame_checks:
    QUESTION_SPECS.append({'qid': f"frame_check_{item['id']}", 'group': 'frame_check', 'question': item['question'], 'description': item['description'], 'reference_name': None, 'reference_description': None})


def encode_image_file(path: Path):
    suffix = path.suffix.lower()
    mime = 'image/png' if suffix == '.png' else 'image/jpeg'
    b64 = base64.b64encode(path.read_bytes()).decode()
    return mime, b64


def image_path_to_data_url(image_path: Path):
    mime, b64 = encode_image_file(image_path)
    return f'data:{mime};base64,{b64}'


def normalize_answer(value):
    if value is None:
        return ''
    value = str(value).strip()
    if not value:
        return ''
    return {'yes': 'Yes', 'no': 'No', 'uncertain': 'Uncertain', 'unknown': 'Unknown'}.get(value.lower(), value)


def normalize_tool_label(value):
    if value is None:
        return ''
    value = str(value).strip()
    if not value:
        return ''
    mapping = {'grasper': 'Grasper', 'maryland': 'Maryland', 'hook': 'Hook', 'irrigator': 'Irrigator', 'scissors': 'Scissors', 'clipper': 'Clipper', 'unknown': 'Unknown', '(absent)': '(absent)', '(not set)': '(absent)', 'not set': '(absent)', 'none': '(absent)'}
    return mapping.get(value.lower(), value)


def build_prompt_signature(prompt_record):
    canonical = json.dumps(prompt_record, sort_keys=True, ensure_ascii=True)
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()


def build_text_prompt_record(text, label=None):
    record = {'type': 'text', 'text': text}
    if label:
        record['label'] = label
    return record


def build_image_prompt_record(path, label, reference_name=None, reference_description=None):
    record = {'type': 'image', 'label': label, 'path': str(Path(path)), 'sha256': hashlib.sha256(Path(path).read_bytes()).hexdigest()}
    if reference_name is not None:
        record['reference_name'] = reference_name
    if reference_description is not None:
        record['reference_description'] = reference_description
    return record


reference_images = {}
for name, path in REFERENCE_IMAGE_PATHS.items():
    mime, b64 = encode_image_file(path)
    reference_images[name] = {'path': str(path), 'mime': mime, 'b64': b64}


In [3]:
records = load_audit_records(AUDIT_V11_DIR)
gt_simple_records = [convert_record_to_simple_actions(record) for record in records]
source_results = read_json(SOURCE_ARTIFACT_DIR / 'results.json', {'results': []})['results']
record_by_example = {record['example_id']: record for record in records}


def sampled_frame_ids(frame_range, frame_stride):
    start_frame, end_frame = [int(x) for x in frame_range]
    frame_ids = list(range(start_frame, end_frame + 1, int(frame_stride)))
    if not frame_ids:
        frame_ids = [start_frame]
    if frame_ids[-1] != end_frame:
        frame_ids.append(end_frame)
    return sorted(dict.fromkeys(frame_ids))


def collect_clip_frame_paths(video_id, frame_range, frame_stride=FRAME_STRIDE):
    frame_ids = sampled_frame_ids(frame_range, frame_stride)
    paths = []
    for frame_id in frame_ids:
        path = FRAMES_DIR / video_id / f'frame_{int(frame_id):06d}.png'
        if path.exists():
            paths.append(str(path))
    return paths


def clip_tool_from_segments(segments):
    tools = sorted({
        normalize_tool_label(seg.get('tool_type'))
        for seg in (segments or [])
        if normalize_tool_label(seg.get('tool_type'))
    })
    if not tools:
        return '(absent)', '(absent)'
    if len(tools) == 1:
        return tools[0], tools[0]
    return '(multi)', ' | '.join(tools)


interval_rows = []
for row in source_results:
    if row.get('status') != 'ok':
        continue
    for right_idx, seg in enumerate((row.get('extracted_actions') or {}).get('right', []) or []):
        start_frame = int(seg.get('start_frame'))
        end_frame = int(seg.get('end_frame'))
        image_paths = collect_clip_frame_paths(row['video_id'], [start_frame, end_frame], FRAME_STRIDE)
        interval_rows.append({
            'interval_key': f"{row['example_id']}::{row['method']}::right_{right_idx:02d}_{start_frame:06d}_{end_frame:06d}",
            'example_id': row['example_id'],
            'video_id': row['video_id'],
            'criterion': row.get('criterion'),
            'method': row['method'],
            'right_index': right_idx,
            'frame_range': [start_frame, end_frame],
            'source_tool_type': normalize_tool_label(seg.get('tool_type')),
            'source_action_code': seg.get('action_code'),
            'source_target_structure': seg.get('target_structure'),
            'image_paths': image_paths,
            'n_images': len(image_paths),
        })

interval_df = pd.DataFrame(interval_rows)
print('interval rows:', len(interval_df))
display(interval_df.head())


interval rows: 561


,interval_key,example_id,video_id,criterion,method,right_index,frame_range,source_tool_type,source_action_code,source_target_structure,image_paths,n_images
0,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2,C1,hint_questions_cvs_context,0,"[1950, 2070]",Hook,DISSECT,CysticArtery,[/mnt/md0/weiqiuy/surgent/data/processed/CVS_C...,5
1,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2,C1,hint_questions_cvs_context,1,"[2130, 2400]",Hook,DISSECT,CysticArtery,[/mnt/md0/weiqiuy/surgent/data/processed/CVS_C...,10
2,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2,C1,structured_prediction_cvs_context,0,"[1950, 2100]",Hook,DISSECT,CysticArtery,[/mnt/md0/weiqiuy/surgent/data/processed/CVS_C...,6
3,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2,C1,structured_prediction_cvs_context,1,"[2100, 2250]",Hook,DISSECT,CysticArtery,[/mnt/md0/weiqiuy/surgent/data/processed/CVS_C...,6
4,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2__C1__avg_...,00467596-8200-449c-8528-d4816ec2f6a2,C1,structured_prediction_deterministic_cvs_context,0,"[1950, 2070]",Hook,DISSECT,CysticArtery,[/mnt/md0/weiqiuy/surgent/data/processed/CVS_C...,5


In [4]:
def _build_questionnaire_prompt():
    question_blocks = []
    for item in QUESTION_SPECS:
        block = f"- {item['qid']}: {item['question']}"
        if item.get('description'):
            block += f"\n  Yes definition: {item['description'].get('yes', '')}"
            block += f"\n  No definition: {item['description'].get('no', '')}"
            if item['description'].get('uncertain'):
                block += f"\n  Uncertain definition: {item['description'].get('uncertain', '')}"
        block += '\n  Allowed answers: Yes, No, Uncertain'
        question_blocks.append(block)

    return f"""
You are a surgical instrument identification assistant.
You are viewing one or more frames from the same predicted right-hand action interval from a laparoscopic cholecystectomy clip.
Your task is to answer the exact questionnaire below about the actively working RIGHT-side instrument.

Rules:
- Answer every question with exactly one of: Yes, No, Uncertain.
- For the 10 instrument questions, judge visible instrument morphology only.
- For the 3 frame-check questions, judge presence and visibility only.
- Use all provided frames together when they help resolve ambiguity.
- If the active working instrument is not clearly present, use Uncertain for morphology questions when needed.
- Do not skip any question.

Questions:
{chr(10).join(question_blocks)}

Return STRICT JSON only with this schema:
{{
  "answers": {{
    "instrument_1": "Yes|No|Uncertain",
    "instrument_2": "Yes|No|Uncertain",
    "instrument_3": "Yes|No|Uncertain",
    "instrument_4": "Yes|No|Uncertain",
    "instrument_5": "Yes|No|Uncertain",
    "instrument_6": "Yes|No|Uncertain",
    "instrument_7": "Yes|No|Uncertain",
    "instrument_8": "Yes|No|Uncertain",
    "instrument_9": "Yes|No|Uncertain",
    "instrument_10": "Yes|No|Uncertain",
    "frame_check_1": "Yes|No|Uncertain",
    "frame_check_2": "Yes|No|Uncertain",
    "frame_check_3": "Yes|No|Uncertain"
  }},
  "brief_reasoning": "1-3 short sentences"
}}
""".strip()


def build_interval_prompt_record(interval_row):
    questionnaire_prompt = _build_questionnaire_prompt()
    user_record_content = [build_text_prompt_record(questionnaire_prompt, label='questionnaire')]
    for item in instrument_rubric:
        ref = reference_images[item['reference_name']]
        ref_text = f"Reference for instrument_{item['id']}: {item['question']}\nYes definition: {item['description'].get('yes', '')}\nNo definition: {item['description'].get('no', '')}\nUncertain definition: {item['description'].get('uncertain', '')}"
        user_record_content.append(build_text_prompt_record(ref_text, label=f"reference_text_instrument_{item['id']}"))
        user_record_content.append(build_image_prompt_record(ref['path'], label=f"reference_image_instrument_{item['id']}", reference_name=item['reference_name'], reference_description=item['reference_description']))
    for idx, image_path in enumerate(interval_row['image_paths']):
        user_record_content.append(build_image_prompt_record(image_path, label=f"interval_frame_{idx:02d}"))
    return {
        'method': 'prompt2_all_in_one_json_with_shapes',
        'prompt_version': PROMPT_VERSION,
        'model': MODEL_ID,
        'temperature': REQUEST_TEMPERATURE,
        'enable_thinking': REQUEST_ENABLE_THINKING,
        'interval_key': interval_row['interval_key'],
        'example_id': interval_row['example_id'],
        'source_method': interval_row['method'],
        'source_tool_type': interval_row['source_tool_type'],
        'frame_range': interval_row['frame_range'],
        'messages': [
            {'role': 'system', 'content': 'You answer surgical instrument questionnaires from images. Return strict JSON only.'},
            {'role': 'user', 'content': user_record_content},
        ],
    }


def build_interval_messages(interval_row):
    questionnaire_prompt = _build_questionnaire_prompt()
    user_content = [{'type': 'text', 'text': questionnaire_prompt}]
    for item in instrument_rubric:
        ref = reference_images[item['reference_name']]
        ref_text = f"Reference for instrument_{item['id']}: {item['question']}\nYes definition: {item['description'].get('yes', '')}\nNo definition: {item['description'].get('no', '')}\nUncertain definition: {item['description'].get('uncertain', '')}"
        user_content.append({'type': 'text', 'text': ref_text})
        user_content.append({'type': 'image_url', 'image_url': {'url': f"data:{ref['mime']};base64,{ref['b64']}"}})
    for image_path in interval_row['image_paths']:
        user_content.append({'type': 'image_url', 'image_url': {'url': image_path_to_data_url(Path(image_path))}})
    return [
        {'role': 'system', 'content': 'You answer surgical instrument questionnaires from images. Return strict JSON only.'},
        {'role': 'user', 'content': user_content},
    ]


def compute_normalized_tool_scores(answer_map):
    tool_breakdown = {}
    for tool in TOOL_TYPES:
        scored_items = []
        earned_points = 0
        total_possible = 0
        for item in instrument_rubric:
            expected, points = item['scores'].get(tool, (None, 0))
            if expected is None or points <= 0:
                continue
            total_possible += points
            observed = normalize_answer(answer_map.get(f"instrument_{item['id']}", 'Uncertain'))
            expected = normalize_answer(expected)
            matched = observed == expected
            earned = points if matched else 0
            earned_points += earned
            scored_items.append({'question_id': f"instrument_{item['id']}", 'expected': expected, 'observed': observed, 'points': points, 'matched': matched, 'earned': earned})
        normalized_score = (earned_points / total_possible) if total_possible else 0.0
        tool_breakdown[tool] = {'earned_points': earned_points, 'total_possible': total_possible, 'normalized_score': normalized_score, 'items': scored_items}
    sorted_tools = sorted(((tool, info['normalized_score']) for tool, info in tool_breakdown.items()), key=lambda item: (-item[1], item[0]))
    return {'tool_breakdown': tool_breakdown, 'tool_scores': {tool: info['normalized_score'] for tool, info in tool_breakdown.items()}, 'sorted_tools': sorted_tools, 'predicted_tool': sorted_tools[0][0] if sorted_tools else 'Unknown'}


def aggregate_tool_prediction(answer_map):
    return compute_normalized_tool_scores(answer_map)


def load_interval_cache():
    if INTERVAL_CACHE_PATH.is_file():
        return json.loads(INTERVAL_CACHE_PATH.read_text())
    return {}


def save_interval_cache(cache):
    INTERVAL_CACHE_PATH.write_text(json.dumps(cache, indent=2))


def interval_cache_is_current(entry, interval_row):
    if not isinstance(entry, dict):
        return False
    if entry.get('prompt_version') != PROMPT_VERSION:
        return False
    expected_signature = build_prompt_signature(build_interval_prompt_record(interval_row))
    return entry.get('prompt_signature') == expected_signature


def run_interval_rubric(interval_row):
    messages = build_interval_messages(interval_row)
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        temperature=REQUEST_TEMPERATURE,
        extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
    )
    raw_text = response.choices[0].message.content
    parsed = extract_json_object(raw_text)
    answers = {q['qid']: normalize_answer(parsed.get('answers', {}).get(q['qid'], 'Uncertain')) for q in QUESTION_SPECS}
    return {'raw': raw_text, 'answers': answers, 'aggregation': aggregate_tool_prediction(answers)}


In [5]:
interval_cache = load_interval_cache()
processed_rows = []

for interval_row in tqdm(interval_rows, desc='Running interval rubric', dynamic_ncols=True):
    cache_key = interval_row['interval_key']
    existing = interval_cache.get(cache_key)
    if interval_cache_is_current(existing, interval_row) and not FORCE_INTERVAL_RUBRIC:
        processed = existing
    else:
        if not interval_row['image_paths']:
            processed = {
                'raw': '',
                'answers': {},
                'aggregation': {'predicted_tool': ''},
                'prompt_version': PROMPT_VERSION,
                'prompt_record': build_interval_prompt_record(interval_row),
                'prompt_signature': build_prompt_signature(build_interval_prompt_record(interval_row)),
                'skipped_reason': 'no_image_paths',
            }
        else:
            processed = run_interval_rubric(interval_row)
            processed['prompt_version'] = PROMPT_VERSION
            processed['prompt_record'] = build_interval_prompt_record(interval_row)
            processed['prompt_signature'] = build_prompt_signature(processed['prompt_record'])
        interval_cache[cache_key] = processed
        save_interval_cache(interval_cache)

    processed_rows.append({
        **interval_row,
        'rubric_predicted_tool': normalize_tool_label((processed.get('aggregation') or {}).get('predicted_tool', '')),
        'rubric_sorted_tools': (processed.get('aggregation') or {}).get('sorted_tools', []),
    })

if WRITE_DERIVED_ARTIFACTS:
    write_json(INTERVAL_ROWS_PATH, processed_rows)

processed_df = pd.DataFrame(processed_rows)
display(processed_df[['method', 'source_tool_type', 'rubric_predicted_tool', 'criterion', 'n_images']].value_counts().reset_index(name='count'))
print('saved interval cache:', INTERVAL_CACHE_PATH)
print('saved interval rows:', INTERVAL_ROWS_PATH)


Running interval rubric:   0%|                                                                 | 0/561 [00:00<…

,method,source_tool_type,rubric_predicted_tool,criterion,n_images,count
0,structured_prediction_cvs_context,Hook,Clipper,C1,3,7
1,structured_prediction_deterministic_cvs_context,Hook,Clipper,C1,3,7
2,hint_questions_cvs_context,Hook,Scissors,C2,11,5
3,description_only_cvs_context,Hook,Scissors,C2,11,5
4,hint_questions_cvs_context,Hook,Irrigator,C1,5,4
...,...,...,...,...,...,...
372,structured_prediction_cvs_context,Hook,Scissors,C1,9,1
373,structured_prediction_deterministic_cvs_context,Hook,Scissors,C1,9,1
374,hint_questions_cvs_context,Hook,Hook,C2,16,1
375,structured_prediction_cvs_context,Hook,Scissors,C2,9,1


saved interval cache: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_2__debug5/right_interval_rubric_cache.json
saved interval rows: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_2__debug5/right_interval_rubric_rows.json


In [6]:
rubric_by_interval = {row['interval_key']: row for row in processed_rows}

derived_results = []
override_rows = []

for row in source_results:
    item = deepcopy(row)
    if item.get('status') != 'ok':
        derived_results.append(item)
        continue

    applied_segments = []
    right_segments = (item.get('extracted_actions') or {}).get('right', []) or []
    for right_idx, seg in enumerate(right_segments):
        start_frame = int(seg.get('start_frame'))
        end_frame = int(seg.get('end_frame'))
        interval_key = f"{item['example_id']}::{item['method']}::right_{right_idx:02d}_{start_frame:06d}_{end_frame:06d}"
        interval_row = rubric_by_interval.get(interval_key, {})
        rubric_tool = normalize_tool_label(interval_row.get('rubric_predicted_tool', ''))
        original_tool = normalize_tool_label(seg.get('tool_type'))
        applied = False
        if original_tool == 'Hook' and rubric_tool == 'Maryland':
            seg['tool_type'] = 'Maryland'
            triplet = list(seg.get('triplet') or [])
            if triplet:
                triplet[0] = 'Maryland'
                seg['triplet'] = triplet
            applied = True
        applied_segments.append({
            'interval_key': interval_key,
            'right_index': right_idx,
            'frame_range': [start_frame, end_frame],
            'source_tool_type': original_tool,
            'rubric_predicted_tool': rubric_tool,
            'override_applied': applied,
        })

    item['extracted_actions_natural_language'] = naturalize_simple_actions(item['extracted_actions'])
    item['labeling_version'] = LABELING_VERSION
    item['v3_2_override'] = {
        'source_result_dir': str(SOURCE_ARTIFACT_DIR),
        'interval_rubric_cache_path': str(INTERVAL_CACHE_PATH),
        'interval_rubric_rows_path': str(INTERVAL_ROWS_PATH),
        'rule': 'for each right interval: if original tool is Hook and interval rubric predicts Maryland, rewrite interval tool to Maryland',
        'segments': applied_segments,
    }
    derived_results.append(item)

    for seg_meta in applied_segments:
        override_rows.append({
            'example_id': item['example_id'],
            'video_id': item['video_id'],
            'criterion': item.get('criterion'),
            'method': item['method'],
            **seg_meta,
        })

if WRITE_DERIVED_ARTIFACTS:
    write_json(RESULTS_PATH, {'results': derived_results})
    write_json(SUMMARY_PATH, override_rows)

override_df = pd.DataFrame(override_rows)
display(override_df.groupby(['method', 'source_tool_type', 'rubric_predicted_tool'], dropna=False)['override_applied'].agg(['count', 'sum']).reset_index().sort_values(['method', 'sum', 'count'], ascending=[True, False, False]))
print('saved results:', RESULTS_PATH)


,method,source_tool_type,rubric_predicted_tool,count,sum
6,description_only_cvs_context,Hook,Maryland,23,23
3,description_only_cvs_context,Hook,Clipper,34,0
4,description_only_cvs_context,Hook,Hook,23,0
7,description_only_cvs_context,Hook,Scissors,17,0
5,description_only_cvs_context,Hook,Irrigator,10,0
8,description_only_cvs_context,Irrigator,Irrigator,3,0
11,description_only_cvs_context,Maryland,Maryland,3,0
10,description_only_cvs_context,Maryland,Clipper,2,0
0,description_only_cvs_context,Clipper,Clipper,1,0
1,description_only_cvs_context,Clipper,Hook,1,0


saved results: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_2__debug5/results.json


In [7]:
ok_results = [item for item in derived_results if item.get('status') == 'ok']
methods = sorted({item.get('method') for item in ok_results})
spec_eval = {}
for method_name in methods:
    pred_records = []
    for item in ok_results:
        if item['method'] != method_name:
            continue
        pred = dict(item['extracted_actions'])
        pred['example_id'] = item['example_id']
        pred['video_id'] = item['video_id']
        pred['criterion'] = item.get('criterion')
        pred_records.append(pred)
    spec_eval[method_name] = evaluate_method_predictions(pred_records, gt_simple_records)

if WRITE_DERIVED_ARTIFACTS:
    write_json(SPEC_EVAL_PATH, spec_eval)

rows = []
for method_name, method_rows in spec_eval.items():
    for row in method_rows:
        rows.append({k: v for k, v in {'method': method_name, **row}.items() if k != 'details'})

spec_df = pd.DataFrame(rows)
overall = (
    spec_df.groupby('method', as_index=False)
    .agg(mean_f1=('f1_mean', 'mean'), rows=('f1_mean', 'size'))
    .sort_values('mean_f1', ascending=False)
)
actor = (
    spec_df.groupby(['actor', 'method'], as_index=False)
    .agg(mean_f1=('f1_mean', 'mean'))
    .pivot(index='actor', columns='method', values='mean_f1')
    .reset_index()
)

display(overall)
display(actor)


,method,mean_f1,rows
2,structured_prediction_cvs_context,0.329243,30
1,hint_questions_cvs_context,0.282669,30
3,structured_prediction_deterministic_cvs_context,0.282100,30
0,description_only_cvs_context,0.207342,30


method,actor,description_only_cvs_context,hint_questions_cvs_context,structured_prediction_cvs_context,structured_prediction_deterministic_cvs_context
0,camera,0.016314,0.130742,0.167994,0.153000
1,left,0.143423,0.413602,0.587771,0.555643
2,other,1.000000,0.555556,0.333333,0.000000
3,right,0.198069,0.212700,0.230601,0.231689
